### GMM intensity clustering example

Gaussian Mixture Models group pixels by intensity. This educational demo visualizes intensity clusters in brain CT images; the clusters are not validated hemorrhage labels. Bright structures can include skull and other tissue. Clinical detection claims require annotated data and evaluation.


In [ ]:
#load required libraries
import os
import numpy as np
import matplotlib.pyplot as plt

#GMM model from sklearn
from sklearn.mixture import GaussianMixture

#image processing functions from the associated skimage library
from skimage import io, color

### Define a function to load images from local folder

In [ ]:
def load_image_as_array(path):
    """Read a grayscale, RGB, or RGBA image as a floating-point grayscale array."""
    from skimage.util import img_as_float

    img = io.imread(path)
    if img.ndim == 2:
        return img_as_float(img)
    if img.ndim == 3 and img.shape[-1] in (3, 4):
        if img.shape[-1] == 4:
            img = color.rgba2rgb(img)
        return color.rgb2gray(img)
    raise ValueError(f"Expected grayscale, RGB, or RGBA image, got shape {img.shape}")


### Define a function to apply GMM model to loaded images

In [ ]:
def segment_image_with_gmm(img_array, n_components):
    """Segment the image using Gaussian Mixture Model (GMM)."""
    img_array = np.asarray(img_array)
    if img_array.ndim != 2 or img_array.size == 0 or not np.isfinite(img_array).all():
        raise ValueError("Expected a nonempty finite 2D grayscale image")
    if (not isinstance(n_components, (int, np.integer)) or isinstance(n_components, bool)
            or not 1 <= n_components <= np.unique(img_array).size):
        raise ValueError("n_components must be an integer between 1 and the number of distinct intensities")
    h, w = img_array.shape
    pixels = img_array.reshape(-1, 1) #reshape the image to a 2D array of pixels

    # fit GMM
    gmm = GaussianMixture(n_components=n_components, covariance_type='tied', random_state=42, verbose=1)
    gmm.fit(pixels)
    labels = gmm.predict(pixels) #get labels for each pixel

    # sort cluster indices by mean intensity (easier to visually interpret)
    means = gmm.means_.flatten() #get cluster mean
    sorted_indices = np.argsort(means)  # low intensity -> high
    label_map = np.zeros_like(sorted_indices) #create mapping
    label_map[sorted_indices] = np.arange(n_components)

    # map labels to sorted indices
    sorted_labels = label_map[labels]

    # reshape to original image
    segmented = sorted_labels.reshape(h, w)
    
    return segmented


### Define a function to display the results

In [ ]:
def show_segmentation(original, segmented):
    """Display the original and segmented images side by side."""
    plt.figure(figsize=(10, 5))

    #show the original image in grayscale
    plt.subplot(1, 2, 1)
    plt.imshow(original, cmap='gray') #specify grayscale so matplotlib doesn't use a colormap
    plt.title('Original')
    plt.axis('off')

    # show the segmented image with a colormap assigned by the GMM
    plt.subplot(1, 2, 2)
    plt.imshow(segmented, cmap='viridis')
    plt.title('GMM Segmentation')
    plt.axis('off')

    plt.tight_layout()
    plt.show()


### Test our functions!

Images: https://www.kaggle.com/datasets/felipekitamura/head-ct-hemorrhage

Follow README.md to download with the Kaggle API outside the checkout. Set `GMM_IMAGE_DIR` to the extracted dataset directory. PNG files are discovered recursively.


In [ ]:
from pathlib import Path

folder = Path(os.environ.get("GMM_IMAGE_DIR", "/workspace/datasets/head-ct-hemorrhage")).expanduser()
if not folder.is_dir():
    raise FileNotFoundError(f"Image directory does not exist: {folder}. Download the dataset (see README) or set GMM_IMAGE_DIR.")
image_files = sorted(p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() == ".png")
if not image_files:
    raise FileNotFoundError(f"No PNG images found under {folder}. Check the download and GMM_IMAGE_DIR.")

for img_path in image_files:
    img = load_image_as_array(img_path)
    segmented = segment_image_with_gmm(img, n_components=5)
    show_segmentation(img, segmented)


### Overlay intensity clusters on the original images

The brightest cluster is not a validated anomaly or hemorrhage mask.


In [ ]:
def overlay_mask_skimage(image, mask, target_label=None, alpha=0.4, bg_label=0):
    """
    Overlay GMM mask onto grayscale image using skimage's label2rgb.
    """
    from skimage.color import label2rgb

    if target_label is not None:
        mask = (mask == target_label).astype(int)
    else:
        mask = mask.astype(int)
    
    # label2rgb overlays a color-coded label map on top of a grayscale image
    overlay = label2rgb(mask, image=image, alpha=alpha, bg_label=bg_label, image_alpha=1.0)

    # Plot the result
    plt.figure(figsize=(10, 5))
    plt.imshow(overlay)
    plt.title(f"Overlay (Label {target_label if target_label is not None else 'All'})")
    plt.axis('off')
    plt.show()


### Let's test the overlay function on the same images

In [ ]:
for img_path in image_files:
    img = load_image_as_array(img_path)
    segmented = segment_image_with_gmm(img, n_components=5)
    # Label 4 is the brightest of five intensity clusters, not a hemorrhage label.
    overlay_mask_skimage(img, segmented, target_label=4, alpha=0.5)


### References:

- Contreras Carrasco, O. (2024). Gaussian Mixture Model Explained. Builtin.com. https://builtin.com/articles/gaussian-mixture-model
- Kitamura, F. (2018). Head CT - hemorrhage. Kaggle.com. https://www.kaggle.com/datasets/felipekitamura/head-ct-hemorrhage
- Raieli, S. (2024, April 10). Image Segmentation with Simple and Elegant Methods. Medium; Python in Plain English. https://python.plainenglish.io/image-segmentation-with-simple-and-elegant-methods-2c5920024b61
- Talasila, A. (2020, October). Generating Image Segmentation Masks - The Easy Way. Towards Data Science. https://towardsdatascience.com/generating-image-segmentation-masks-the-easy-way-dd4d3656dbd1/